# Graph End-to-End Validation Notebook
Este notebook executa os testes de uso das classes novas em sequência:
1. Carrega `.env` e autentica no Graph
2. Obtém e apresenta o conteúdo do site
3. Lista arquivos do drive e salva em DataFrame (`df_drive_items`)
4. Executa fluxo de arquivo: write, update e load
5. Executa testes de listas (views, colunas, itens, create e update)

In [ ]:
from __future__ import annotations
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Garante carregamento do .env a partir da raiz do repositório
repo_root = Path.cwd()
if not (repo_root / '.env').exists() and (repo_root.parent / '.env').exists():
    repo_root = repo_root.parent

env_path = repo_root / '.env'
load_dotenv(env_path)

print(f'.env carregado de: {env_path}')
print('Pronto para iniciar os testes.')

In [ ]:
from msgraphclient.auth import GraphClient
from msgraphclient.drive import GraphDrive
from msgraphclient.lists import GraphList

client = GraphClient()
print('Cliente Graph inicializado com sucesso.')

site_attributes = {
    'sharepoint_site_id': client.authenticator.sharepoint_site_id,
    'site_graph_id': client.site_graph_id,
    'site_name': client.site_name,
    'site_display_name': client.site_display_name,
    'site_web_url': client.site_web_url,
}

print('\nAtributos do site conectado:')
for key, value in site_attributes.items():
    print(f'- {key}: {value}')

In [ ]:
site_contents = client.get_site_contents()
site_data = site_contents.get('site', {})
drives_data = site_contents.get('drives', [])
lists_data = site_contents.get('lists', [])

print('Resumo do conteúdo do site:')
print(f"- Site: {site_data.get('displayName', site_data.get('name', '-'))}")
print(f"- Drives encontrados: {len(drives_data)}")
print(f"- Lists encontradas: {len(lists_data)}")

df_site_drives = pd.json_normalize(drives_data) if drives_data else pd.DataFrame()
df_site_lists = pd.json_normalize(lists_data) if lists_data else pd.DataFrame()

print('\nDrives Disponíveis:')
display(df_site_drives.head())

print('Lists Disponíveis:')
display(df_site_lists.head())

In [ ]:
import os
drive_id = os.environ["SHAREPOINT_DRIVE_ID"]
drive = GraphDrive(drive_id=drive_id, client=client)
drive_items = drive.list_drive_items()

df_drive_items = pd.json_normalize(drive_items) if drive_items else pd.DataFrame()
num_files = len([item for item in drive_items if 'folder' not in item])

print(f'Total de itens no drive root: {len(drive_items)}')
print(f'Total de arquivos (sem pastas): {num_files}')
print('\nDados disponíveis:')

display(df_drive_items.head())

## File Write, Update and Load
Este bloco cria (ou reutiliza) um arquivo de teste, atualiza o conteúdo e lê novamente para validação.

In [ ]:
test_local_file = repo_root / 'notebooks' / 'downloads' / 'notebook_test_upload.txt'
test_local_file.parent.mkdir(parents=True, exist_ok=True)
test_local_file.write_text(
    f'Notebook initial content - {datetime.now(timezone.utc).isoformat()}\\n',
    encoding='utf-8',
)

upload_result = drive.upload_file(
    local_path=test_local_file,
    remote_folder='root',
    remote_name=f'notebook_test_upload_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}.txt',
)
target_item_id = str(upload_result.get('id', ''))

if not target_item_id:
    raise RuntimeError('Falha ao obter item id após upload do arquivo de teste.')

print('Write concluído (upload):')
print(f"- Item ID: {target_item_id}")
print(f"- Nome: {upload_result.get('name')}")

updated_content = (
    f'Notebook updated content - {datetime.now(timezone.utc).isoformat()}\\n'
    'Linha adicional para teste de update.\\n'
    'Fim.\\n'
 )
update_result = drive.write_file_content(target_item_id, updated_content)

print('Update concluído:')
print(f"- Item ID retornado: {update_result.get('id')}")

loaded_content = drive.read_file_content(target_item_id)
print('Load concluído (conteúdo atual lido):')
print(loaded_content)

download_path = repo_root / 'notebooks' / 'downloads' / f'downloaded_{target_item_id}.txt'
saved_path = drive.download_file(target_item_id, download_path)
print(f'Arquivo carregado para disco local em: {saved_path}')

## List Tests (Guided)
Este bloco agora segue um fluxo guiado:
1. Inicializa `GraphList` com compatibilidade de versão da API.
2. Exibe schema, tipos e templates de payload (fáceis de copiar/colar e editar).
3. Permite salvar um item como `create` ou `update` em uma célula separada.

In [ ]:
import importlib
import json
import msgraphclient.lists as lists_mod

# Recarrega módulo para evitar cache de versões anteriores no kernel.
importlib.reload(lists_mod)
GraphList = lists_mod.GraphList

import os
list_id = os.environ["SHAREPOINT_LIST_ID"]
list_client = GraphList(list_id=list_id, client=client)

# Compatibilidade entre nomes novos e antigos (caso o ambiente esteja desatualizado).
get_views_fn = getattr(list_client, 'get_views', None) or getattr(list_client, 'get_list_views')
get_columns_fn = getattr(list_client, 'get_columns', None) or getattr(list_client, 'get_list_columns')
get_view_columns_fn = getattr(list_client, 'get_view_columns', None) or getattr(list_client, 'get_list_view_columns', None)

views = get_views_fn()
columns = get_columns_fn()
schema = list_client.get_schema()
field_types = list_client.get_field_types()

print('GraphList inicializado com sucesso.')
print(f'- Método views ativo: {get_views_fn.__name__}')
print(f'- Método columns ativo: {get_columns_fn.__name__}')
if get_view_columns_fn:
    print(f'- Método view_columns ativo: {get_view_columns_fn.__name__}')

print(f'Views encontradas: {len(views)}')
print(f'Colunas consultadas: {len(columns)}')
print(f'Colunas no schema validado: {len(schema)}')

df_schema = pd.DataFrame(schema) if schema else pd.DataFrame()
print('\nSchema validado da lista (display_name, type, required, read_only):')
display(df_schema)

df_list_items = list_client.get_items_dataframe(include_id=True)
print(f'Itens atuais encontrados: {len(df_list_items)}')
display(df_list_items.head())

schema_by_name = {entry['display_name']: entry for entry in schema}
now = datetime.now(timezone.utc)

def first_writable_of_type(field_type: str) -> str | None:
    for entry in schema:
        if entry.get('read_only'):
            continue
        if entry.get('type') == field_type:
            return str(entry['display_name'])
    return None

# Template mínimo para create (campos obrigatórios).
suggested_payload_create = list_client.get_item_template(include_optional=False)
for display_name in list(suggested_payload_create.keys()):
    col_type = field_types.get(display_name)
    if col_type in ('text', 'note', 'personOrGroup'):
        suggested_payload_create[display_name] = f'Novo item - {now.strftime("%Y-%m-%d %H:%M:%S")}'
    elif col_type == 'number':
        suggested_payload_create[display_name] = 0
    elif col_type == 'boolean':
        suggested_payload_create[display_name] = False
    elif col_type == 'dateTime':
        suggested_payload_create[display_name] = now.isoformat()
    elif col_type == 'choice':
        choices = schema_by_name.get(display_name, {}).get('choices', [])
        suggested_payload_create[display_name] = choices[0] if choices else None

# Template exemplo para update (usa primeiro _id encontrado, se existir).
suggested_payload_update = {'_id': str(df_list_items.iloc[0]['_id'])} if not df_list_items.empty else {'_id': '<INSIRA_O_ID_AQUI>'}
text_col = first_writable_of_type('text')
if text_col:
    suggested_payload_update[text_col] = f'Atualizado via notebook em {now.isoformat()}'
number_col = first_writable_of_type('number')
if number_col:
    suggested_payload_update[number_col] = 123.45
boolean_col = first_writable_of_type('boolean')
if boolean_col:
    suggested_payload_update[boolean_col] = True
date_col = first_writable_of_type('dateTime')
if date_col:
    suggested_payload_update[date_col] = now.isoformat()
choice_col = first_writable_of_type('choice')
if choice_col:
    choices = schema_by_name.get(choice_col, {}).get('choices', [])
    if choices:
        suggested_payload_update[choice_col] = choices[0]

print('\nTemplate sugerido para CREATE (copie e edite na próxima célula):')
print(json.dumps(suggested_payload_create, ensure_ascii=False, indent=2, default=str))

print('\nTemplate sugerido para UPDATE (copie e edite na próxima célula):')
print(json.dumps(suggested_payload_update, ensure_ascii=False, indent=2, default=str))

### Save Operation (Editável)
Edite os valores da próxima célula e execute:
- `operation = 'create'` para inserir novo item.
- `operation = 'update'` para atualizar item existente (exige `_id`).

In [ ]:
# Escolha: 'create' ou 'update'
operation = 'update'

# Copie um dos templates exibidos na célula anterior e edite aqui.
payload = dict(
    {
        "_id": "1",
        "Title": "Atualizado via notebook em 2026-05-28T15:00:05.792790+00:00",
        "measurementCount": 123.45,
    }
)

# Exemplo opcional para create:
# operation = 'create'
# payload = dict(suggested_payload_create)

if operation not in ('create', 'update'):
    raise ValueError("operation deve ser 'create' ou 'update'")

if operation == 'create':
    payload.pop('_id', None)
else:
    if '_id' not in payload or not str(payload['_id']).strip():
        raise ValueError("Para update, informe payload['_id'] com o ID do item")

print('Validando payload...')
list_client.validate_item(payload)

print(f'Salvando item ({operation})...')
saved_item = list_client.save_item(payload)
print('Operação concluída com sucesso:')
print(saved_item)

print('\nAmostra atualizada de itens:')
df_list_items_after_save = list_client.get_items_dataframe(include_id=True)
display(df_list_items_after_save.head())